# Exploring Cached Presto Embeddings
This notebook queries the DuckDB cache at `embeddings_cache_LANDCOVER10.duckdb` and loads embeddings into pandas for inspection.

In [ ]:
# Imports and path setup
import duckdb
import pandas as pd
from pathlib import Path

# Path to the DuckDB embeddings cache — adjust to your local setup
DB_PATH = Path('/path/to/embeddings_cache_LANDCOVER10.duckdb')
DB_PATH.exists()

In [ ]:
# Open connection and list tables
con = duckdb.connect(str(DB_PATH))
con.execute('SHOW TABLES').fetchdf()

In [ ]:
# Basic row counts and distinct model hashes
summary_df = con.execute("""
SELECT model_hash,
       COUNT(*) AS n_rows,
       COUNT(DISTINCT sample_id) AS n_samples
FROM embeddings_cache
GROUP BY model_hash
ORDER BY n_rows DESC
""").fetchdf()
summary_df

In [ ]:
# Preview first 5 rows (wide format)
preview_df = con.execute('SELECT * FROM embeddings_cache LIMIT 5').fetchdf()
preview_df.head()

In [ ]:
# Load a sample of embeddings as vectors (stack embedding_0..embedding_127)
sample_df = con.execute('SELECT * FROM embeddings_cache LIMIT 100').fetchdf()
embedding_cols = [c for c in sample_df.columns if c.startswith('embedding_')]
sample_df['embedding_vector'] = sample_df[embedding_cols].values.tolist()
sample_df[['sample_id','model_hash','embedding_vector']].head()

In [ ]:
# Compute mean embedding per model_hash (on a sample for speed)
# NOTE: Adjust LIMIT or remove it for full-table stats if memory permits.
df_sample = con.execute('SELECT * FROM embeddings_cache LIMIT 2000').fetchdf()
embedding_cols = [c for c in df_sample.columns if c.startswith('embedding_')]
mean_embeddings = df_sample.groupby('model_hash')[embedding_cols].mean()
mean_embeddings.head()

In [ ]:
from worldcereal.utils.refdata import get_class_mappings
from worldcereal.utils.refdata import map_classes

CLASS_MAPPINGS = get_class_mappings()

# Path to the merged wide-format parquet — adjust to your local setup
SOURCE_PARQUET = '/path/to/worldcereal_all_extractions_wide_month_LANDCOVER10.parquet'
df_src = pd.read_parquet(SOURCE_PARQUET, columns=['sample_id','ewoc_code'])
df_src = map_classes(df_src, 'LANDCOVER10', class_mappings=CLASS_MAPPINGS)
df_src.head()

In [ ]:
# Configuration for class embedding visualization (using raw ewoc_code)
EMB_DB = str(DB_PATH)   # reuse path from the first cell
SAMPLE_PER_CLASS = 300  # reduce/increase for speed vs fidelity
RANDOM_SEED = 42

import duckdb, pandas as pd, numpy as np, math, os, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

# Load source labels (raw ewoc_code)
code_counts = df_src["finetune_class"].value_counts()

# Balanced sampling per ewoc_code
selected_ids = []
for code in code_counts.index.tolist():
    ids = df_src.loc[df_src.finetune_class==code,'sample_id']
    if len(ids) > SAMPLE_PER_CLASS:
        ids = ids.sample(SAMPLE_PER_CLASS, random_state=RANDOM_SEED)
    selected_ids.extend(ids.tolist())

print(f'Total sampled ids: {len(selected_ids)}')

con = duckdb.connect(EMB_DB)
con.register('sel_ids', pd.DataFrame({'sample_id': selected_ids}))
# Fetch embeddings for only selected ids
emb_cols = ', '.join([f'embedding_{i}' for i in range(128)])
query = f"""
SELECT e.sample_id, e.model_hash, {emb_cols}
FROM embeddings_cache e
INNER JOIN sel_ids USING(sample_id)
"""
emb_df = con.execute(query).fetchdf()

# Join ewoc_code labels
emb_df = emb_df.merge(df_src, on='sample_id', how='left')
print('Embeddings shape:', emb_df.shape)
emb_df.head()

In [ ]:
# Prepare embedding matrix and labels (ewoc_code)
embedding_matrix = emb_df[[f'embedding_{i}' for i in range(128)]].to_numpy(dtype=np.float32)
labels = emb_df['finetune_class'].astype(str)
class_to_int = {c:i for i,c in enumerate(sorted(set(labels)))}
label_ids = np.array([class_to_int[l] for l in labels], dtype=np.int32)
print('Matrix shape:', embedding_matrix.shape, 'Num ewoc_codes:', len(class_to_int))

# Standardize (z-score) to equalize scale per dimension
mean = embedding_matrix.mean(axis=0, keepdims=True)
std = embedding_matrix.std(axis=0, keepdims=True) + 1e-6
emb_std = (embedding_matrix - mean) / std

In [ ]:
# PCA 2D & 3D projection (ewoc_code labeling)
from sklearn.decomposition import PCA

pca = PCA(n_components=3, random_state=42)
pca3 = pca.fit_transform(emb_std)
pca2 = pca3[:, :2]

pca_df = pd.DataFrame({'pc1': pca2[:,0], 'pc2': pca2[:,1], 'finetune_class': labels})
pca3_df = pd.DataFrame({'pc1': pca3[:,0], 'pc2': pca3[:,1], 'pc3': pca3[:,2], 'ewoc_code': labels})
pca_df.head()

In [ ]:
# 2D scatter plot (Matplotlib) of PCA by finetune_class
import matplotlib.pyplot as plt
plt.figure(figsize=(7,6))
for code in sorted(class_to_int.keys()):
    mask = labels == code
    plt.scatter(pca_df.loc[mask,'pc1'], pca_df.loc[mask,'pc2'], s=10, alpha=0.6, label=code)
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('PCA 2D Projection by finetune_class')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: t-SNE (subset) by finetune_class
from sklearn.manifold import TSNE
MAX_TSNE = 3000
idx = np.arange(len(emb_std))
if len(idx) > MAX_TSNE:
    idx = np.random.choice(idx, MAX_TSNE, replace=False)
tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42)
tsne_coords = tsne.fit_transform(emb_std[idx])
tsne_labels = labels[idx]
import matplotlib.pyplot as plt
plt.figure(figsize=(7,6))
for code in sorted(class_to_int.keys()):
    mask = tsne_labels == code
    plt.scatter(tsne_coords[mask,0], tsne_coords[mask,1], s=10, alpha=0.6, label=code)
plt.title('t-SNE 2D (subset) by finetune_class')
plt.xlabel('Dim 1'); plt.ylabel('Dim 2'); plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout(); plt.show()


In [ ]:
# Cluster quality metrics using ewoc_code
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
X_cluster = pca2  # reduced 2D
k = len(class_to_int)
km = KMeans(n_clusters=k, random_state=42, n_init='auto').fit(X_cluster)
sil = silhouette_score(X_cluster, km.labels_)
print(f'Silhouette score (k={k}): {sil:.4f}')
# Distance to ewoc_code mean in standardized space
code_means = {code: emb_std[labels==code].mean(0) for code in class_to_int.keys()}
per_sample_dist = np.empty(len(emb_std), dtype=np.float32)
for i, code in enumerate(labels):
    per_sample_dist[i] = np.linalg.norm(emb_std[i] - code_means[code])
dist_df = pd.DataFrame({'sample_id': emb_df.sample_id, 'ewoc_code': labels, 'dist_to_code_mean': per_sample_dist})
print(dist_df.head())
print('Per-code mean distance (lower => tighter cluster):')
print(dist_df.groupby('ewoc_code')['dist_to_code_mean'].mean().sort_values())

In [ ]:
# UMAP 2D projection (subset for speed)
UMAP_MAX = 8000  # tune for performance

import umap

umap_idx = np.arange(len(emb_std))
if len(umap_idx) > UMAP_MAX:
    umap_idx = np.random.choice(umap_idx, UMAP_MAX, replace=False)
X_umap = emb_std[umap_idx]
labels_umap = labels[umap_idx]
print('Running UMAP on', X_umap.shape)
reducer = umap.UMAP(n_components=2, random_state=42, metric='euclidean', n_neighbors=30, min_dist=0.15)
umap_coords = reducer.fit_transform(X_umap)

umap_df = pd.DataFrame({'u1': umap_coords[:,0], 'u2': umap_coords[:,1], 'finetune_class': labels_umap})
umap_df.head()

In [ ]:
# Interactive Plotly scatter for UMAP
import plotly.express as px

color_count = umap_df['finetune_class'].nunique()
# If too many classes, show top frequent and bucket others as 'OTHER'
MAX_LEGEND = 30
if color_count > MAX_LEGEND:
    top_codes = umap_df['finetune_class'].value_counts().head(MAX_LEGEND).index
    umap_df['finetune_class_plot'] = np.where(umap_df['finetune_class'].isin(top_codes), umap_df['finetune_class'], 'OTHER')
else:
    umap_df['finetune_class_plot'] = umap_df['finetune_class']

fig = px.scatter(
    umap_df,
    x='u1', y='u2',
    color='finetune_class_plot',
    hover_data={'finetune_class': True, 'u1': ':.3f', 'u2': ':.3f'},
    title=f'UMAP projection ({len(umap_df)} samples)'
)
fig.update_layout(legend=dict(itemsizing='trace', title='finetune_class'))
fig.show()